# Question to judged candidate

One question runs through the agents. Each cell shows one step.

`question -> plan -> KG reading -> figure reading -> design -> geometry -> tool calls -> verdict -> report`

Section 1 adds three design fields that tie a bioinspired mechanism from the KG to a printed ceramic. The tools are the stand-ins in `tools/`, and [../tools/README.md](../tools/README.md) gives what a real tool must match.

Start jupyter with the model server up and `CORPUS` set to a corpus folder written by `pipeline/scripts/build_corpus.py`.

In [ ]:
import json
import os
import sys
import time
from pathlib import Path

# Imports and run folders are relative to the repository root, the first folder up that holds tools/.
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tools" / "schema.py").exists())
os.chdir(root)
sys.path.insert(0, str(root))

from IPython.display import Markdown

from pipeline import corpus, graph
from pipeline.llm import parse

c = corpus.load(os.environ["CORPUS"], graph.EMBEDDING)
figures = int(c["is_figure"].sum())
print(len(c["chunks"]) - figures, "passages,", figures, "figures,", len(c["triples"]), "triples")
graph.MODELS  # model and reasoning effort per agent

## 1. Design fields

The first three fields state the bioinspired hypothesis, the biological model, its toughening mechanism and the additive manufacturing process. The designer picks each from the KG and the papers, with a source. The KG is asked about every option, and it ranks first the triples that link two options, such as nacre and crack deflection.

In [ ]:
{field: spec.get("enum", spec["description"]) for field, spec in graph.DESIGN.items()}

## 2. Question

`state` holds what every agent reads and writes. The run folder receives one folder per candidate.

In [ ]:
question = ("Propose a bioinspired layered ceramic made by additive manufacturing whose interfaces deflect cracks. "
            "Choose the biological model and its mechanism, the ceramic, the interlayer, the print process and the "
            "layer thicknesses, and simulate the design.")
folder = Path("runs") / time.strftime("%Y%m%d-%H%M%S")
app = graph.build(graph.MODELS, c)
start = time.perf_counter()
for mode, chunk in app.stream({"question": question, "folder": str(folder)}, {"recursion_limit": 100},
                              stream_mode=["updates", "values"]):
    if mode == "updates":
        print(f"{time.perf_counter() - start:6.1f} s  {', '.join(chunk)}")
    else:
        state = chunk

### Plan
Search queries and KG entities from the question.

In [ ]:
parse(state["trace"][0]["reply"]["content"])

### KG reading
Retrieved passages and figures, plus KG triples for the entities and every design option. The triples listed link two options. They carry the mechanism from the biological model to the ceramic.

In [ ]:
passages = [p for p in state["passages"] if "image" not in p]
print(len(passages), "passages,", len(state["passages"]) - len(passages), "figures,", len(state["triples"]), "triples")
options = [o.lower() for o in graph.OPTIONS]
print("\nKG triples that link two options")
for t in state["triples"]:
    hit = [o for o in options if o in f"{t['head']} {t['relation']} {t['tail']}".lower()]
    if len(hit) > 1:
        print(" ", hit, "|", t["relation"][:95], "|", t["source"])

### Figure reading
The values the vision model read from each figure, with the place in the figure it read each one.

In [ ]:
for n in state["figure_notes"]:
    print(n["id"], [(v.get("quantity"), v.get("value"), v.get("unit"), v.get("where")) for v in n["reading"].get("values", [])])

### Design
One value per field of the last candidate, each with its source. The checks reject a value that its source does not state, and the designer gets one repair. A field that still fails stops the run before any tool call.

In [ ]:
for field in state["design"]:
    print(f"{field:24} {str(state['design'][field]):18} {state['sources'].get(field, '')[:90]}")
print("missing:", state["missing"])
print("rejected:", json.dumps(state["rejected"], indent=1))

### Geometry
The STL files of the part and the description the FEM tool receives.

In [ ]:
geometry = json.load(open(state["geometry"]["geometry_json"]))
for part in ["bar", "cell"]:
    print(part, geometry[part]["size_mm"], "mm,", geometry[part]["ceramic_layers"], "ceramic layers,",
          list(geometry[part]["files"].values()))
print("notch", geometry["cell"]["notch"])

### Tool calls and handoff files
The engineer's calls, in order. Each call is recorded in the handoff file of its tool, with the origin of every input.

In [ ]:
for r in state["results"]:
    print(f"{r['tool']}({json.dumps(r['arguments'])})\n   -> {r['output']}")
    print("   origins:", {k: i["from"] for k, i in (r["inputs"] or {}).items()}, "\n")
print(sorted(p.name for p in Path(state["candidate"], "handoff").iterdir()))

### Verdict, novelty and reward
The judge's finding on each criterion for every candidate, with its novelty against the corpus and its reward. A candidate the judge sends back, or an accepted one below the novelty target, returns the designer to work.

In [ ]:
for candidate in state.get("candidates", []):
    v = candidate["verdict"]
    print(candidate["candidate"], v["verdict"], "(provisional)" * bool(v.get("provisional")),
          f"novelty {candidate['novelty']['score']:.2f}, reward {candidate['reward']:.2f} |", v.get("summary"))
    for k in v.get("criteria", []):
        print("   ", k["met"], "|", k["criterion"], "|", k["evidence"])
print("\n" + state["outcome"])

### Recipe and print files
The fabricator's recipe for the best candidate. Each number cites the passage that states it, or is marked as an assumption. Code turns the geometry into the print files of the chosen process.

In [ ]:
if state.get("recipe"):
    print(open(Path(state["folder"]) / state["best"] / "recipe.md").read())
    print(json.dumps(state["fabrication"]["files"], indent=1))

### Report

In [ ]:
Markdown(state["report"])